# Stretch: six more charts, worked solutions

The answers to the stretch notebook. Try each one before looking.

## Setup

In [ ]:
import os
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

here = Path.cwd()
while not (here / "data" / "clean" / "plays.csv").exists() and here != here.parent:
    here = here.parent
os.chdir(here)
os.makedirs("output", exist_ok=True)

plays = pd.read_csv("data/clean/plays.csv")
plays["date"] = pd.to_datetime(plays["played_at"])

BLUE, GREY, HIGHLIGHT = "#0072b2", "#b8bec9", "#d55e00"
print(len(plays), "plays")

## Stretch 1: an honest skip-rate chart

In [ ]:
rates = plays.groupby("genre")["skipped"].agg(plays="size", skips="sum", rate="mean")
rates["rate"] = rates["rate"] * 100
rates = rates.sort_values("rate")                 # barh draws from the bottom up

top_genre = rates.index[-1]
colours = [HIGHLIGHT if genre == top_genre else GREY for genre in rates.index]
labels = [f"{rate:.1f}% of {n}" for rate, n in zip(rates["rate"], rates["plays"])]

fig, ax = plt.subplots(figsize=(8, 4))
bars = ax.barh(rates.index, rates["rate"], color=colours)
ax.bar_label(bars, labels=labels, padding=4)
ax.set_title(f"{top_genre} is skipped most: {rates['rate'].iloc[-1]:.1f}% of its plays")
ax.set_xlabel("plays skipped (%)")
ax.set_xlim(0, 21)
ax.spines[["top", "right"]].set_visible(False)

fig.savefig("output/skip_rate_by_genre.png", dpi=150, bbox_inches="tight")
plt.show()

assert top_genre == "Jazz"
assert (rates.loc["Jazz", "plays"], rates.loc["Jazz", "skips"]) == (154, 24)

`zip` walks two columns side by side, one pair at a time, which is what you
need to put two numbers in each label. Labelling each bar with its play
count lets the reader judge for themselves how much to trust each rate.

**Is Jazz really skipped more?** Count what the title is built on:

In [ ]:
overall = plays["skipped"].mean()
expected = rates.loc["Jazz", "plays"] * overall
print(f"overall skip rate: {overall:.1%}")
print(f"Jazz at that rate would have {expected:.1f} skips; it has {rates.loc['Jazz', 'skips']}")

assert round(overall * 100, 1) == 11.8
assert round(expected, 1) == 18.2

About six skips more than an average genre would have. With 154 plays,
that is the kind of difference chance produces all the time. The chart is
honest because the labels show the 154; the title would be more honest
still as "Jazz is skipped most, by a handful of plays". There are tests for
exactly this question, called significance tests. They are beyond this
course, but now you know what question they answer.

## Stretch 2: draw the wrong dates

In [ ]:
messy = pd.read_csv("data/messy/plays_messy.csv", keep_default_na=False)
deduped = messy.drop_duplicates().sort_values("play_id")
guessed = pd.to_datetime(deduped["played_at"], format="mixed").dt.strftime("%Y-%m-%d")

clean_sorted = plays.sort_values("play_id")
wrong_rows = (guessed.values != clean_sorted["played_at"].values).sum()

right = clean_sorted["played_at"].str[:7].value_counts().sort_index()
wrong = guessed.str[:7].value_counts().sort_index()

fig, ax = plt.subplots(figsize=(10, 4.5))
ax.plot(pd.to_datetime(right.index), right.values, marker="o", color=BLUE,
        label="each format parsed on purpose")
ax.plot(pd.to_datetime(wrong.index), wrong.values, marker="o", color=HIGHLIGHT,
        label='format="mixed"')
ax.set_title(f'format="mixed" puts {wrong_rows} plays in the wrong place')
ax.set_ylabel("plays per month")
ax.set_ylim(bottom=0)
ax.legend()
ax.spines[["top", "right"]].set_visible(False)
plt.show()

In [ ]:
moved = (wrong - right).sort_values()
print(moved.head(2))
print(moved.tail(2))

assert len(deduped) == 2183
assert wrong_rows == 158
assert moved["2025-01"] == -8 and moved["2025-12"] == 7

January 2025 lost 8 plays and December 2025 gained 7. That is what a
day/month swap does: the 12th of January becomes the 1st of December, and
so on. Plays are not lost, they are moved, so the total for the year stays
right and every monthly number is a little bit off.

**Could you spot it from the wrong line alone?** No. It still dips every
summer, it still rises in autumn. Nothing about it looks broken. That is
the whole danger of a silent wrong parse, and why session 6 parsed each
date format on purpose.

Two lines means a legend, or a direct label. With one line, the title does
that job.

## Stretch 3: a slope chart

In [ ]:
per_year = plays.pivot_table(index="genre", columns=plays["date"].dt.year,
                             values="play_id", aggfunc="count")
per_year["change"] = per_year[2025] - per_year[2024]
fell, rose = per_year["change"].idxmin(), per_year["change"].idxmax()

fig, ax = plt.subplots(figsize=(6, 6))
for genre, row in per_year.iterrows():
    if genre in (fell, rose):
        colour, width = (HIGHLIGHT if genre == fell else BLUE), 3
        ax.text(-0.05, row[2024], f"{genre} {row[2024]}", ha="right", va="center", color=colour)
        ax.text(1.05, row[2025], f"{row[2025]} {genre}", ha="left", va="center", color=colour)
    else:
        colour, width = GREY, 1.5
    ax.plot([0, 1], [row[2024], row[2025]], color=colour, linewidth=width, marker="o")

ax.set_xticks([0, 1], ["2024", "2025"])
ax.set_xlim(-0.6, 1.6)
ax.set_ylim(bottom=0)
ax.set_ylabel("plays")
ax.set_title(f"{fell} fell by {-per_year.loc[fell, 'change']} plays, "
             f"{rose} rose by {per_year.loc[rose, 'change']}")
ax.spines[["top", "right", "bottom"]].set_visible(False)
plt.show()

assert (fell, rose) == ("Folk", "Afrobeat")
assert (per_year.loc["Folk", "change"], per_year.loc["Afrobeat", "change"]) == (-34, 14)

A slope chart is a line chart with only two points in time. The slope of
each line is the change, so steep means a big change and flat means none.
Every genre is there in grey for scale; only the two in the title get
colour and labels.

Why label only two? Try labelling all eight: on the left, Rock (69), Jazz
(72) and Afrobeat (83) sit so close that their labels overlap. Labelling
what the title talks about is a choice, not a shortcut.

The x-axis is just the numbers 0 and 1 with year labels on top, which is
all a slope chart needs.

## Stretch 4: noise, and a rolling average

In [ ]:
weekly = plays.set_index("date").resample("W").size()
print(weekly.tail(3))
print("last week runs to", weekly.index[-1].date(), "but the data ends on",
      plays["date"].max().date())

`resample("W")` makes weeks that end on a Sunday. The last one ends on
4 January 2026, and the data stops on 31 December: that week has three days
in it. Left in, it would show as a sudden drop at the end that never
happened. Drop it.

In [ ]:
weekly = weekly.iloc[:-1]
smooth = weekly.rolling(4).mean()

fig, ax = plt.subplots(figsize=(10, 4.5))
ax.plot(weekly.index, weekly.values, color=GREY, linewidth=1, label="plays per week")
ax.plot(smooth.index, smooth.values, color=BLUE, linewidth=3, label="average of the last 4 weeks")
ax.set_title(f"Week to week it jumps between {weekly.min()} and {weekly.max()} plays; "
             "the average shows the seasons")
ax.set_ylabel("plays per week")
ax.set_ylim(bottom=0)
ax.legend(frameon=False)
ax.spines[["top", "right"]].set_visible(False)
plt.show()

assert len(weekly) == 104
assert (weekly.min(), weekly.max()) == (8, 35)
assert round(smooth.min(), 1) == 12.0
assert f"{smooth.idxmin():%B %Y}" == "July 2025"

`.rolling(4).mean()` replaces each week with the average of that week and
the three before it. The first three weeks have no value, because there are
not yet four weeks to average. The jumpy grey line is still there, so the
reader can see what was smoothed away.

**What the smooth line does not show:** any single week. The quietest
stretch, mid July 2025 at 12.0 plays a week, is real; the single quietest
week, 8 plays, has vanished into it. Smoothing is a choice, and the window
(4 weeks here) is part of that choice.

## Stretch 5: a heatmap

In [ ]:
grid = plays.pivot_table(index=plays["date"].dt.dayofweek, columns=plays["date"].dt.month,
                         values="play_id", aggfunc="count")

day_names = ["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"]
month_names = ["Jan", "Feb", "Mar", "Apr", "May", "Jun",
               "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"]

fig, ax = plt.subplots(figsize=(11, 4.5))
image = ax.imshow(grid.values, cmap="Blues", vmin=0)
fig.colorbar(image, ax=ax, label="plays, both years together")

for row in range(grid.shape[0]):
    for column in range(grid.shape[1]):
        value = grid.values[row, column]
        ax.text(column, row, value, ha="center", va="center",
                color="white" if value > 28 else "black")

ax.set_xticks(range(12), month_names)
ax.set_yticks(range(7), day_names)
ax.set_title("The month matters far more than the day of the week")
plt.show()

In [ ]:
by_day = grid.sum(axis=1)
by_month = grid.sum(axis=0)
print(f"days of the week: {by_day.min()} to {by_day.max()}, "
      f"busiest {by_day.max() / by_day.min():.2f} times the quietest")
print(f"months:           {by_month.min()} to {by_month.max()}, "
      f"busiest {by_month.max() / by_month.min():.2f} times the quietest")

assert (by_day.min(), by_day.max()) == (302, 333)
assert (by_month.min(), by_month.max()) == (120, 239)

The busiest month has about twice the plays of the quietest; the busiest
weekday only 10% more than the quietest. The colour shows it at a glance:
the columns change from light to dark, the rows hardly at all.

* `imshow` draws a grid of numbers as colours
* `cmap="Blues"` goes from light to dark in one colour, so darker is always
  more. A rainbow colour map has no such order
* `vmin=0` starts the colours at zero, the heatmap version of bars starting
  at zero
* the numbers are written in, because nobody can read 23 against 25 from
  two shades of blue. White on dark cells, black on light ones

**What it does not show:** single days. Each cell adds up about eight or
nine days across two years, so one long Sunday would vanish into it.

## Stretch 6: the same small multiples, in seaborn

seaborn is a library built on matplotlib that draws common statistical
charts in one call. It is **not** in the course requirements, so you may
need to install it first, in a terminal:

```
pip install seaborn
```

The cell below says so and stops politely if it is missing.

In [ ]:
plays["quarter"] = plays["date"].dt.to_period("Q").dt.start_time
long = plays.groupby(["genre", "quarter"]).size().reset_index(name="plays")
print(long.head())

try:
    import seaborn as sns
except ImportError:
    sns = None
    print("seaborn is not installed: run  pip install seaborn  and try again")

if sns is not None:
    grid = sns.relplot(data=long, x="quarter", y="plays", col="genre", col_wrap=4,
                       kind="line", height=2.4, aspect=1.3, color=BLUE)
    grid.set_titles("{col_name}")
    grid.set_axis_labels("", "plays per quarter")
    grid.set(ylim=(0, None))
    for ax in grid.axes.flat:
        ax.set_xticks([pd.Timestamp("2024-01-01"), pd.Timestamp("2025-01-01")], ["2024", "2025"])
    grid.figure.suptitle("Electronic and Folk are the two biggest genres in every quarter", y=1.03)
    plt.show()

assert len(long) == 64

seaborn wants **long** data: one row per genre per quarter, with columns
for each thing on the chart, rather than the wide table `pivot_table` made
in notebook 02. `groupby(...).size().reset_index(name="plays")` makes it.

Then one call: `col="genre"` makes one chart per genre, `col_wrap=4` wraps
them into rows of four, and the y-axis is shared by default.
`grid.set(ylim=(0, None))` starts every chart at zero and leaves the top
to seaborn. Compare it with
the loop in notebook 02. seaborn is quicker for the standard charts;
matplotlib is what you drop down to when you need something it does not
do, and seaborn hands you the matplotlib `ax` objects (`grid.axes`) for
exactly that.